In [1]:
import pandas as pd
import numpy as np

In [2]:
from google.colab import files
uploaded = files.upload()

Saving KDDTrain+ (1).txt to KDDTrain+ (1).txt


In [3]:
from google.colab import files
uploaded = files.upload()

Saving KDDTest+ (1).txt to KDDTest+ (1).txt


In [4]:
columns = [
"duration","protocol_type","service","flag","src_bytes","dst_bytes",
"land","wrong_fragment","urgent","hot","num_failed_logins",
"logged_in","num_compromised","root_shell","su_attempted",
"num_root","num_file_creations","num_shells","num_access_files",
"num_outbound_cmds","is_host_login","is_guest_login","count",
"srv_count","serror_rate","srv_serror_rate","rerror_rate",
"srv_rerror_rate","same_srv_rate","diff_srv_rate","srv_diff_host_rate",
"dst_host_count","dst_host_srv_count","dst_host_same_srv_rate",
"dst_host_diff_srv_rate","dst_host_same_src_port_rate",
"dst_host_srv_diff_host_rate","dst_host_serror_rate",
"dst_host_srv_serror_rate","dst_host_rerror_rate",
"dst_host_srv_rerror_rate","label","difficulty"
]

In [5]:
train = pd.read_csv("KDDTrain+ (1).txt", names = columns)
test = pd.read_csv("KDDTest+ (1).txt", names = columns)

In [6]:
train.head()

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [7]:
train.shape

(125973, 43)

In [8]:
train.head()


,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,...,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label,difficulty
0,0,tcp,ftp_data,SF,491,0,0,0,0,0,...,0.17,0.03,0.17,0.00,0.00,0.00,0.05,0.00,normal,20
1,0,udp,other,SF,146,0,0,0,0,0,...,0.00,0.60,0.88,0.00,0.00,0.00,0.00,0.00,normal,15
2,0,tcp,private,S0,0,0,0,0,0,0,...,0.10,0.05,0.00,0.00,1.00,1.00,0.00,0.00,neptune,19
3,0,tcp,http,SF,232,8153,0,0,0,0,...,1.00,0.00,0.03,0.04,0.03,0.01,0.00,0.01,normal,21
4,0,tcp,http,SF,199,420,0,0,0,0,...,1.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,normal,21


In [9]:
train.shape

(125973, 43)

In [10]:
train['label'].value_counts()

,count
label,
normal,67343
neptune,41214
satan,3633
ipsweep,3599
portsweep,2931
smurf,2646
nmap,1493
back,956
teardrop,892


In [11]:
train['label'].value_counts().head()

,count
label,
normal,67343
neptune,41214
satan,3633
ipsweep,3599
portsweep,2931


In [12]:
train['label'] = train['label'].apply(lambda x: 0 if str(x).strip() == 'normal' else 1)
test['label'] = test['label'].apply(lambda x: 0 if str(x).strip() == 'normal' else 1)

In [13]:
train['label'].value_counts()

,count
label,
0,67343
1,58630


In [14]:
train.drop('difficulty', axis=1, inplace=True)
test.drop('difficulty', axis=1, inplace=True)

In [15]:
train.shape

(125973, 42)

In [16]:
train = pd.get_dummies(train, columns=['protocol_type','service','flag'])
test = pd.get_dummies(test, columns=['protocol_type','service','flag'])

In [17]:
train.shape

(125973, 123)

In [18]:
X_train = train.drop('label', axis=1)
y_train = train['label']

X_test = test.drop('label', axis=1)
y_test = test['label']

In [19]:
X_train.shape

(125973, 122)

In [20]:
X_train, X_test = X_train.align(X_test, join='left', axis=1, fill_value=0)

In [21]:
from sklearn.preprocessing import MinMaxScaler

In [22]:
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)

In [23]:
X_test = scaler.transform(X_test)


In [24]:
X_train.shape

(125973, 122)

In [25]:
X_train = X_train.reshape(X_train.shape[0], X_train.shape[1], 1)
X_test = X_test.reshape(X_test.shape[0], X_test.shape[1], 1)

X_train.shape


(125973, 122, 1)

In [26]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout

In [27]:
model = Sequential()

model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], X_train.shape[2])))
model.add(MaxPooling1D(pool_size=2))
model.add(LSTM(64))
model.add(Dropout(0.5))
model.add(Dense(32, activation='relu'))
model.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [28]:
model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

In [29]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d (Conv1D)                 │ (None, 120, 64)        │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 60, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ (None, 64)             │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 35,393 (138.25 KB)

 Trainable params: 35,393 (138.25 KB)

 Non-trainable params: 0 (0.00 B)

In [30]:
y_train = y_train.astype('float32')
y_test = y_test.astype('float32')
X_train = X_train.astype('float32')
X_test = X_test.astype('float32')

In [31]:
print(type(X_train))
print(type(y_train))
print(type(X_test))
print(type(y_test))

print(X_train.shape)
print(y_train.shape)
print(X_train.dtype)
print(y_train.dtype)

<class 'numpy.ndarray'>
<class 'pandas.core.series.Series'>
<class 'numpy.ndarray'>
<class 'pandas.core.series.Series'>
(125973, 122, 1)
(125973,)
float32
float32


In [32]:
history = model.fit(
    X_train, y_train,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

Epoch 1/5
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 138s 43ms/step - accuracy: 0.9488 - loss: 0.1628 - val_accuracy: 0.9730 - val_loss: 0.0949
Epoch 2/5
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 132s 42ms/step - accuracy: 0.9736 - loss: 0.0890 - val_accuracy: 0.9768 - val_loss: 0.0681
Epoch 3/5
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 143s 42ms/step - accuracy: 0.9765 - loss: 0.0707 - val_accuracy: 0.9800 - val_loss: 0.0571
Epoch 4/5
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 133s 42ms/step - accuracy: 0.9794 - loss: 0.0614 - val_accuracy: 0.9852 - val_loss: 0.0526
Epoch 5/5
3150/3150 ━━━━━━━━━━━━━━━━━━━━ 136s 43ms/step - accuracy: 0.9834 - loss: 0.0517 - val_accuracy: 0.9867 - val_loss: 0.0441


In [33]:
test_loss, test_accuracy = model.evaluate(X_test, y_test)
print("Test Accuracy:", test_accuracy)

705/705 ━━━━━━━━━━━━━━━━━━━━ 10s 14ms/step - accuracy: 0.7282 - loss: 0.8024
Test Accuracy: 0.7282203435897827


In [34]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")

705/705 ━━━━━━━━━━━━━━━━━━━━ 8s 12ms/step


In [35]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report

In [36]:
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1 Score:", f1_score(y_test, y_pred))

Precision: 0.9205970898143502
Recall: 0.5718849840255591
F1 Score: 0.7055034847392454


In [37]:
cm = confusion_matrix(y_test, y_pred)
print(cm)

[[9078  633]
 [5494 7339]]


In [38]:
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

         0.0       0.62      0.93      0.75      9711
         1.0       0.92      0.57      0.71     12833

    accuracy                           0.73     22544
   macro avg       0.77      0.75      0.73     22544
weighted avg       0.79      0.73      0.72     22544



In [39]:
import tensorflow as tf

In [40]:
loss_object = tf.keras.losses.BinaryCrossentropy()

def fgsm_attack(model, x, y, epsilon):
    x_tensor = tf.convert_to_tensor(x, dtype=tf.float32)
    y_tensor = tf.convert_to_tensor(y, dtype=tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(x_tensor)
        prediction = model(x_tensor)
        loss = loss_object(y_tensor, prediction)

    gradient = tape.gradient(loss, x_tensor)
    signed_grad = tf.sign(gradient)

    x_adv = x_tensor + epsilon * signed_grad
    x_adv = tf.clip_by_value(x_adv, 0, 1)

    return x_adv.numpy()

In [41]:
X_test_small = X_test[:500]
y_test_small = y_test[:500]
epsilon = 0.01
X_test_fgsm = fgsm_attack(model, X_test_small, y_test_small, epsilon)

In [42]:
fgsm_loss, fgsm_accuracy = model.evaluate(X_test_fgsm, y_test_small)
print("FGSM Test Accuracy:", fgsm_accuracy)

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 24ms/step - accuracy: 0.7580 - loss: 0.7996
FGSM Test Accuracy: 0.7580000162124634


In [43]:
y_pred_fgsm = (model.predict(X_test_fgsm) > 0.5).astype("int32").ravel()

16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 21ms/step


In [44]:
from sklearn.metrics import confusion_matrix, precision_score, recall_score, f1_score, classification_report

print("FGSM Precision:", precision_score(y_test_small, y_pred_fgsm))
print("FGSM Recall:", recall_score(y_test_small, y_pred_fgsm))
print("FGSM F1-score:", f1_score(y_test_small, y_pred_fgsm))

cm_fgsm = confusion_matrix(y_test_small, y_pred_fgsm)
print(cm_fgsm)

FGSM Precision: 0.890625
FGSM Recall: 0.6309963099630996
FGSM F1-score: 0.7386609071274298
[[208  21]
 [100 171]]


In [45]:
attack_indices = (y_test_small == 1)
predicted_attacks = y_pred_fgsm[attack_indices]

esr = ((predicted_attacks == 0).sum() / len(predicted_attacks)) * 100
print("FGSM ESR:", esr)

FGSM ESR: 36.90036900369004


In [46]:
batch_size = 128
epsilon = 0.01

fgsm_batches = []

for i in range(0, len(X_test), batch_size):
    x_batch = X_test[i:i+batch_size]
    y_batch = y_test[i:i+batch_size]

    x_adv_batch = fgsm_attack(model, x_batch, y_batch, epsilon)
    fgsm_batches.append(x_adv_batch)

    if i % 2000 == 0:
        print(f"Processed {i} samples")

X_test_fgsm_full = np.concatenate(fgsm_batches, axis=0).astype("float32")

Processed 0 samples
Processed 16000 samples


In [47]:
fgsm_loss_full, fgsm_accuracy_full = model.evaluate(X_test_fgsm_full, y_test, verbose=0)
print("Full FGSM Test Accuracy:", fgsm_accuracy_full)

Full FGSM Test Accuracy: 0.7164212465286255


In [48]:
y_pred_fgsm_full = (model.predict(X_test_fgsm_full, batch_size=128, verbose=0) > 0.5).astype("int32").ravel()

In [49]:
fgsm_precision_full = precision_score(y_test, y_pred_fgsm_full)
fgsm_recall_full = recall_score(y_test, y_pred_fgsm_full)
fgsm_f1_full = f1_score(y_test, y_pred_fgsm_full)
cm_fgsm_full = confusion_matrix(y_test, y_pred_fgsm_full)

print("FGSM Precision:", fgsm_precision_full)
print("FGSM Recall:", fgsm_recall_full)
print("FGSM F1-score:", fgsm_f1_full)
print(cm_fgsm_full)

FGSM Precision: 0.9057459677419355
FGSM Recall: 0.5601184446349255
FGSM F1-score: 0.6921854687274303
[[8963  748]
 [5645 7188]]


In [50]:
attack_mask = (y_test == 1)
esr_full = ((y_pred_fgsm_full[attack_mask] == 0).sum() / attack_mask.sum()) * 100
print("Full FGSM ESR:", esr_full)

Full FGSM ESR: 43.98815553650744


In [51]:
fgsm_results = pd.DataFrame([{
    "Condition": "Baseline CNN-LSTM on FGSM test",
    "Epsilon": epsilon,
    "Accuracy": fgsm_accuracy_full,
    "Precision": fgsm_precision_full,
    "Recall": fgsm_recall_full,
    "F1-score": fgsm_f1_full,
    "ESR": esr_full
}])

fgsm_results

,Condition,Epsilon,Accuracy,Precision,Recall,F1-score,ESR
0,Baseline CNN-LSTM on FGSM test,0.01,0.716421,0.905746,0.560118,0.692185,43.988156


In [52]:
np.save("X_test_fgsm_full.npy", X_test_fgsm_full)
np.save("y_pred_fgsm_full.npy", y_pred_fgsm_full)

In [53]:
def pgd_attack(model, x, y, epsilon=0.01, alpha=0.002, iterations=10):
    x_orig = tf.convert_to_tensor(x, dtype=tf.float32)
    x_adv = tf.identity(x_orig)
    y_tensor = tf.convert_to_tensor(y, dtype=tf.float32)

    for _ in range(iterations):
        with tf.GradientTape() as tape:
            tape.watch(x_adv)
            prediction = model(x_adv)
            loss = loss_object(y_tensor, prediction)

        gradient = tape.gradient(loss, x_adv)
        signed_grad = tf.sign(gradient)

        # small iterative step
        x_adv = x_adv + alpha * signed_grad

        # project back into epsilon-ball around original input
        x_adv = tf.clip_by_value(x_adv, x_orig - epsilon, x_orig + epsilon)

        # keep valid normalized range
        x_adv = tf.clip_by_value(x_adv, 0, 1)

    return x_adv.numpy()


In [54]:
X_test_small = X_test[:500]
y_test_small = y_test[:500]

X_test_pgd = pgd_attack(model, X_test_small, y_test_small, epsilon=0.01, alpha=0.002, iterations=10)
print("PGD subset shape:", X_test_pgd.shape)

PGD subset shape: (500, 122, 1)


In [55]:
pgd_loss, pgd_accuracy = model.evaluate(X_test_pgd, y_test_small, verbose=0)
print("PGD Test Accuracy:", pgd_accuracy)

PGD Test Accuracy: 0.7559999823570251


In [56]:
y_pred_pgd = (model.predict(X_test_pgd, verbose=0) > 0.5).astype("int32").ravel()

print("PGD Precision:", precision_score(y_test_small, y_pred_pgd))
print("PGD Recall:", recall_score(y_test_small, y_pred_pgd))
print("PGD F1-score:", f1_score(y_test_small, y_pred_pgd))

cm_pgd = confusion_matrix(y_test_small, y_pred_pgd)
print(cm_pgd)

PGD Precision: 0.8900523560209425
PGD Recall: 0.6273062730627307
PGD F1-score: 0.7359307359307359
[[208  21]
 [101 170]]


In [57]:
attack_indices = (y_test_small == 1)
predicted_attacks = y_pred_pgd[attack_indices]

pgd_esr = ((predicted_attacks == 0).sum() / len(predicted_attacks)) * 100
print("PGD ESR:", pgd_esr)

PGD ESR: 37.269372693726936


In [58]:
batch_size = 64
pgd_batches = []

for i in range(0, len(X_test), batch_size):
    x_batch = X_test[i:i+batch_size]
    y_batch = y_test[i:i+batch_size]

    x_adv_batch = pgd_attack(model, x_batch, y_batch, epsilon=0.01, alpha=0.002, iterations=10)
    pgd_batches.append(x_adv_batch)

    if i % 2000 == 0:
        print(f"Processed {i} samples")

X_test_pgd_full = np.concatenate(pgd_batches, axis=0).astype("float32")
print("Full PGD shape:", X_test_pgd_full.shape)

Processed 0 samples
Processed 8000 samples
Processed 16000 samples
Full PGD shape: (22544, 122, 1)


In [59]:
pgd_loss_full, pgd_accuracy_full = model.evaluate(X_test_pgd_full, y_test, verbose=0)
print("Full PGD Accuracy:", pgd_accuracy_full)

Full PGD Accuracy: 0.7142033576965332


In [60]:
y_pred_pgd_full = (model.predict(X_test_pgd_full, batch_size=128, verbose=0) > 0.5).astype("int32").ravel()

In [61]:
pgd_precision_full = precision_score(y_test, y_pred_pgd_full)
pgd_recall_full = recall_score(y_test, y_pred_pgd_full)
pgd_f1_full = f1_score(y_test, y_pred_pgd_full)
cm_pgd_full = confusion_matrix(y_test, y_pred_pgd_full)

print("PGD Precision:", pgd_precision_full)
print("PGD Recall:", pgd_recall_full)
print("PGD F1-score:", pgd_f1_full)
print(cm_pgd_full)

PGD Precision: 0.9042257085020243
PGD Recall: 0.5569235564560119
PGD F1-score: 0.6892993200559386
[[8954  757]
 [5686 7147]]


In [62]:
attack_mask = (y_test == 1)
pgd_esr_full = ((y_pred_pgd_full[attack_mask] == 0).sum() / attack_mask.sum()) * 100
print("Full PGD ESR:", pgd_esr_full)


Full PGD ESR: 44.30764435439882


In [63]:
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (125973, 122, 1)
y_train: (125973,)
X_test: (22544, 122, 1)
y_test: (22544,)


In [64]:
model.save("baseline_clean.keras")

In [65]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, LSTM, Dense, Dropout

def build_hybrid_model(input_shape):
    model = Sequential()
    model.add(Conv1D(filters=64, kernel_size=3, activation='relu', input_shape=input_shape))
    model.add(MaxPooling1D(pool_size=2))
    model.add(LSTM(64))
    model.add(Dropout(0.5))
    model.add(Dense(32, activation='relu'))
    model.add(Dense(1, activation='sigmoid'))

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

In [66]:
import tensorflow as tf
import numpy as np

loss_object = tf.keras.losses.BinaryCrossentropy()

def fgsm_attack(model, x, y, epsilon):
    x_tensor = tf.convert_to_tensor(x, dtype=tf.float32)
    y_tensor = tf.convert_to_tensor(y, dtype=tf.float32)

    with tf.GradientTape() as tape:
        tape.watch(x_tensor)
        prediction = model(x_tensor)
        loss = loss_object(y_tensor, prediction)

    gradient = tape.gradient(loss, x_tensor)
    signed_grad = tf.sign(gradient)

    x_adv = x_tensor + epsilon * signed_grad
    x_adv = tf.clip_by_value(x_adv, 0, 1)

    return x_adv.numpy()

In [67]:
batch_size = 128
epsilon_train = 0.01

fgsm_train_batches = []

for i in range(0, len(X_train), batch_size):
    x_batch = X_train[i:i+batch_size]
    y_batch = y_train[i:i+batch_size]

    x_adv_batch = fgsm_attack(model, x_batch, y_batch, epsilon_train)
    fgsm_train_batches.append(x_adv_batch)

    if i % 5000 == 0:
        print(f"Processed {i} training samples")

X_train_fgsm = np.concatenate(fgsm_train_batches, axis=0).astype("float32")

print("X_train_fgsm shape:", X_train_fgsm.shape)

Processed 0 training samples
Processed 80000 training samples
X_train_fgsm shape: (125973, 122, 1)


In [68]:
X_train_mix_fgsm = np.concatenate([X_train, X_train_fgsm], axis=0)
y_train_mix_fgsm = np.concatenate([y_train, y_train], axis=0)

print("Mixed X shape:", X_train_mix_fgsm.shape)
print("Mixed y shape:", y_train_mix_fgsm.shape)

Mixed X shape: (251946, 122, 1)
Mixed y shape: (251946,)


In [69]:
indices = np.random.permutation(len(X_train_mix_fgsm))
X_train_mix_fgsm = X_train_mix_fgsm[indices]
y_train_mix_fgsm = y_train_mix_fgsm[indices]

In [70]:
fgsm_defended_model = build_hybrid_model((X_train.shape[1], X_train.shape[2]))

history_fgsm_def = fgsm_defended_model.fit(
    X_train_mix_fgsm,
    y_train_mix_fgsm,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
6299/6299 ━━━━━━━━━━━━━━━━━━━━ 275s 43ms/step - accuracy: 0.9611 - loss: 0.1198 - val_accuracy: 0.9770 - val_loss: 0.0620
Epoch 2/5
6299/6299 ━━━━━━━━━━━━━━━━━━━━ 324s 43ms/step - accuracy: 0.9790 - loss: 0.0637 - val_accuracy: 0.9851 - val_loss: 0.0467
Epoch 3/5
6299/6299 ━━━━━━━━━━━━━━━━━━━━ 274s 44ms/step - accuracy: 0.9832 - loss: 0.0511 - val_accuracy: 0.9877 - val_loss: 0.0370
Epoch 4/5
6299/6299 ━━━━━━━━━━━━━━━━━━━━ 273s 43ms/step - accuracy: 0.9869 - loss: 0.0414 - val_accuracy: 0.9889 - val_loss: 0.0293
Epoch 5/5
6299/6299 ━━━━━━━━━━━━━━━━━━━━ 289s 46ms/step - accuracy: 0.9908 - loss: 0.0286 - val_accuracy: 0.9937 - val_loss: 0.0206


In [71]:
fgsm_defended_model.save("fgsm_defended_model.keras")

In [72]:
print(type(X_test), getattr(X_test, "shape", None), getattr(X_test, "dtype", None))
print(type(y_test), getattr(y_test, "shape", None), getattr(y_test, "dtype", None))

<class 'numpy.ndarray'> (22544, 122, 1) float32
<class 'pandas.core.series.Series'> (22544,) float32


In [73]:
import numpy as np

X_test_eval = np.asarray(X_test).astype("float32")
y_test_eval = np.asarray(y_test).astype("float32").reshape(-1, 1)

print("X_test_eval:", X_test_eval.shape, X_test_eval.dtype)
print("y_test_eval:", y_test_eval.shape, y_test_eval.dtype)

X_test_eval: (22544, 122, 1) float32
y_test_eval: (22544, 1) float32


In [74]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

y_prob_clean_def = fgsm_defended_model.predict(X_test_eval, batch_size=128, verbose=0).ravel()
y_pred_clean_def = (y_prob_clean_def > 0.5).astype("int32")
y_true_clean_def = y_test_eval.ravel().astype("int32")

In [75]:
clean_acc_def = accuracy_score(y_true_clean_def, y_pred_clean_def)
clean_prec_def = precision_score(y_true_clean_def, y_pred_clean_def)
clean_rec_def = recall_score(y_true_clean_def, y_pred_clean_def)
clean_f1_def = f1_score(y_true_clean_def, y_pred_clean_def)
cm_clean_def = confusion_matrix(y_true_clean_def, y_pred_clean_def)

print("FGSM-defended model Clean Accuracy:", clean_acc_def)
print("FGSM-defended model Clean Precision:", clean_prec_def)
print("FGSM-defended model Clean Recall:", clean_rec_def)
print("FGSM-defended model Clean F1-score:", clean_f1_def)
print(cm_clean_def)

print(classification_report(y_true_clean_def, y_pred_clean_def))

FGSM-defended model Clean Accuracy: 0.7771469127040455
FGSM-defended model Clean Precision: 0.9217889164956249
FGSM-defended model Clean Recall: 0.6649263617236811
FGSM-defended model Clean F1-score: 0.7725667722951561
[[8987  724]
 [4300 8533]]
              precision    recall  f1-score   support

           0       0.68      0.93      0.78      9711
           1       0.92      0.66      0.77     12833

    accuracy                           0.78     22544
   macro avg       0.80      0.80      0.78     22544
weighted avg       0.82      0.78      0.78     22544



In [76]:
X_test_fgsm_eval = np.asarray(X_test_fgsm_full).astype("float32")

In [77]:
y_prob_fgsm_def = fgsm_defended_model.predict(X_test_fgsm_eval, batch_size=128, verbose=0).ravel()
y_pred_fgsm_def = (y_prob_fgsm_def > 0.5).astype("int32")
y_true_fgsm_def = y_test_eval.ravel().astype("int32")

In [78]:
fgsm_acc_def = accuracy_score(y_true_fgsm_def, y_pred_fgsm_def)
fgsm_prec_def = precision_score(y_true_fgsm_def, y_pred_fgsm_def)
fgsm_rec_def = recall_score(y_true_fgsm_def, y_pred_fgsm_def)
fgsm_f1_def = f1_score(y_true_fgsm_def, y_pred_fgsm_def)
cm_fgsm_def = confusion_matrix(y_true_fgsm_def, y_pred_fgsm_def)

print("FGSM-defended model on FGSM Accuracy:", fgsm_acc_def)
print("FGSM-defended model on FGSM Precision:", fgsm_prec_def)
print("FGSM-defended model on FGSM Recall:", fgsm_rec_def)
print("FGSM-defended model on FGSM F1-score:", fgsm_f1_def)
print(cm_fgsm_def)

FGSM-defended model on FGSM Accuracy: 0.8343683463449255
FGSM-defended model on FGSM Precision: 0.9343198090692124
FGSM-defended model on FGSM Recall: 0.7626431855372867
FGSM-defended model on FGSM F1-score: 0.8397974944225158
[[9023  688]
 [3046 9787]]


In [79]:
attack_mask = (y_true_fgsm_def == 1)
fgsm_esr_def = ((y_pred_fgsm_def[attack_mask] == 0).sum() / attack_mask.sum()) * 100
print("FGSM-defended model FGSM ESR:", fgsm_esr_def)

FGSM-defended model FGSM ESR: 23.735681446271332


In [80]:
X_test_pgd_eval = np.asarray(X_test_pgd_full).astype("float32")

In [81]:
y_prob_pgd_def = fgsm_defended_model.predict(X_test_pgd_eval, batch_size=128, verbose=0).ravel()
y_pred_pgd_def = (y_prob_pgd_def > 0.5).astype("int32")
y_true_pgd_def = y_test_eval.ravel().astype("int32")

In [82]:
pgd_acc_def = accuracy_score(y_true_pgd_def, y_pred_pgd_def)
pgd_prec_def = precision_score(y_true_pgd_def, y_pred_pgd_def)
pgd_rec_def = recall_score(y_true_pgd_def, y_pred_pgd_def)
pgd_f1_def = f1_score(y_true_pgd_def, y_pred_pgd_def)
cm_pgd_def = confusion_matrix(y_true_pgd_def, y_pred_pgd_def)

print("FGSM-defended model on PGD Accuracy:", pgd_acc_def)
print("FGSM-defended model on PGD Precision:", pgd_prec_def)
print("FGSM-defended model on PGD Recall:", pgd_rec_def)
print("FGSM-defended model on PGD F1-score:", pgd_f1_def)
print(cm_pgd_def)

FGSM-defended model on PGD Accuracy: 0.8272711142654364
FGSM-defended model on PGD Precision: 0.9342271446614204
FGSM-defended model on PGD Recall: 0.7493181641081587
FGSM-defended model on PGD F1-score: 0.831618092190608
[[9034  677]
 [3217 9616]]


In [83]:
attack_mask = (y_true_pgd_def == 1)
pgd_esr_def = ((y_pred_pgd_def[attack_mask] == 0).sum() / attack_mask.sum()) * 100
print("FGSM-defended model PGD ESR:", pgd_esr_def)

FGSM-defended model PGD ESR: 25.068183589184134


In [84]:
batch_size = 64
epsilon_train = 0.01
alpha_train = 0.002
iterations_train = 10

pgd_train_batches = []

for i in range(0, len(X_train), batch_size):
    x_batch = X_train[i:i+batch_size]
    y_batch = y_train[i:i+batch_size]

    x_adv_batch = pgd_attack(
        model,
        x_batch,
        y_batch,
        epsilon=epsilon_train,
        alpha=alpha_train,
        iterations=iterations_train
    )
    pgd_train_batches.append(x_adv_batch)

    if i % 5000 == 0:
        print(f"Processed {i} training samples")

X_train_pgd = np.concatenate(pgd_train_batches, axis=0).astype("float32")

print("X_train_pgd shape:", X_train_pgd.shape)


Processed 0 training samples
Processed 40000 training samples
Processed 80000 training samples
Processed 120000 training samples
X_train_pgd shape: (125973, 122, 1)


In [85]:
X_train_mix_pgd = np.concatenate([X_train, X_train_pgd], axis=0)
y_train_mix_pgd = np.concatenate([y_train, y_train], axis=0)

print("Mixed PGD X shape:", X_train_mix_pgd.shape)
print("Mixed PGD y shape:", y_train_mix_pgd.shape)

Mixed PGD X shape: (251946, 122, 1)
Mixed PGD y shape: (251946,)


In [86]:
indices = np.random.permutation(len(X_train_mix_pgd))
X_train_mix_pgd = X_train_mix_pgd[indices]
y_train_mix_pgd = y_train_mix_pgd[indices]

In [87]:
pgd_defended_model = build_hybrid_model((X_train.shape[1], X_train.shape[2]))

history_pgd_def = pgd_defended_model.fit(
    X_train_mix_pgd,
    y_train_mix_pgd,
    epochs=5,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Epoch 1/5
6299/6299 ━━━━━━━━━━━━━━━━━━━━ 291s 45ms/step - accuracy: 0.9569 - loss: 0.1276 - val_accuracy: 0.9764 - val_loss: 0.0822
Epoch 2/5
6299/6299 ━━━━━━━━━━━━━━━━━━━━ 322s 45ms/step - accuracy: 0.9765 - loss: 0.0664 - val_accuracy: 0.9796 - val_loss: 0.0528
Epoch 3/5
6299/6299 ━━━━━━━━━━━━━━━━━━━━ 278s 44ms/step - accuracy: 0.9807 - loss: 0.0532 - val_accuracy: 0.9837 - val_loss: 0.0448
Epoch 4/5
6299/6299 ━━━━━━━━━━━━━━━━━━━━ 279s 44ms/step - accuracy: 0.9844 - loss: 0.0445 - val_accuracy: 0.9832 - val_loss: 0.0443
Epoch 5/5
6299/6299 ━━━━━━━━━━━━━━━━━━━━ 365s 51ms/step - accuracy: 0.9879 - loss: 0.0352 - val_accuracy: 0.9920 - val_loss: 0.0252


In [88]:
pgd_defended_model.save("pgd_defended_model.keras")

In [89]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

X_test_eval = np.asarray(X_test).astype("float32")
y_test_eval = np.asarray(y_test).astype("float32").reshape(-1, 1)

y_prob_clean_pgd_def = pgd_defended_model.predict(X_test_eval, batch_size=128, verbose=0).ravel()
y_pred_clean_pgd_def = (y_prob_clean_pgd_def > 0.5).astype("int32")
y_true_clean_pgd_def = y_test_eval.ravel().astype("int32")

In [90]:
clean_acc_pgd_def = accuracy_score(y_true_clean_pgd_def, y_pred_clean_pgd_def)
clean_prec_pgd_def = precision_score(y_true_clean_pgd_def, y_pred_clean_pgd_def)
clean_rec_pgd_def = recall_score(y_true_clean_pgd_def, y_pred_clean_pgd_def)
clean_f1_pgd_def = f1_score(y_true_clean_pgd_def, y_pred_clean_pgd_def)
cm_clean_pgd_def = confusion_matrix(y_true_clean_pgd_def, y_pred_clean_pgd_def)

print("PGD-defended model Clean Accuracy:", clean_acc_pgd_def)
print("PGD-defended model Clean Precision:", clean_prec_pgd_def)
print("PGD-defended model Clean Recall:", clean_rec_pgd_def)
print("PGD-defended model Clean F1-score:", clean_f1_pgd_def)
print(cm_clean_pgd_def)

print(classification_report(y_true_clean_pgd_def, y_pred_clean_pgd_def))

PGD-defended model Clean Accuracy: 0.768319730305181
PGD-defended model Clean Precision: 0.9187761391151221
PGD-defended model Clean Recall: 0.650510402867607
PGD-defended model Clean F1-score: 0.7617135818239883
[[8973  738]
 [4485 8348]]
              precision    recall  f1-score   support

           0       0.67      0.92      0.77      9711
           1       0.92      0.65      0.76     12833

    accuracy                           0.77     22544
   macro avg       0.79      0.79      0.77     22544
weighted avg       0.81      0.77      0.77     22544



In [91]:
X_test_fgsm_eval = np.asarray(X_test_fgsm_full).astype("float32")

y_prob_fgsm_pgd_def = pgd_defended_model.predict(X_test_fgsm_eval, batch_size=128, verbose=0).ravel()
y_pred_fgsm_pgd_def = (y_prob_fgsm_pgd_def > 0.5).astype("int32")
y_true_fgsm_pgd_def = y_test_eval.ravel().astype("int32")

In [92]:
fgsm_acc_pgd_def = accuracy_score(y_true_fgsm_pgd_def, y_pred_fgsm_pgd_def)
fgsm_prec_pgd_def = precision_score(y_true_fgsm_pgd_def, y_pred_fgsm_pgd_def)
fgsm_rec_pgd_def = recall_score(y_true_fgsm_pgd_def, y_pred_fgsm_pgd_def)
fgsm_f1_pgd_def = f1_score(y_true_fgsm_pgd_def, y_pred_fgsm_pgd_def)
cm_fgsm_pgd_def = confusion_matrix(y_true_fgsm_pgd_def, y_pred_fgsm_pgd_def)

print("PGD-defended model on FGSM Accuracy:", fgsm_acc_pgd_def)
print("PGD-defended model on FGSM Precision:", fgsm_prec_pgd_def)
print("PGD-defended model on FGSM Recall:", fgsm_rec_pgd_def)
print("PGD-defended model on FGSM F1-score:", fgsm_f1_pgd_def)
print(cm_fgsm_pgd_def)

PGD-defended model on FGSM Accuracy: 0.7659687721788503
PGD-defended model on FGSM Precision: 0.9170069528749586
PGD-defended model on FGSM Recall: 0.6474713628925427
PGD-defended model on FGSM F1-score: 0.7590207362747785
[[8959  752]
 [4524 8309]]


In [93]:
attack_mask = (y_true_fgsm_pgd_def == 1)
fgsm_esr_pgd_def = ((y_pred_fgsm_pgd_def[attack_mask] == 0).sum() / attack_mask.sum()) * 100
print("PGD-defended model FGSM ESR:", fgsm_esr_pgd_def)

PGD-defended model FGSM ESR: 35.252863710745736


In [94]:
X_test_pgd_eval = np.asarray(X_test_pgd_full).astype("float32")

y_prob_pgd_pgd_def = pgd_defended_model.predict(X_test_pgd_eval, batch_size=128, verbose=0).ravel()
y_pred_pgd_pgd_def = (y_prob_pgd_pgd_def > 0.5).astype("int32")
y_true_pgd_pgd_def = y_test_eval.ravel().astype("int32")

In [95]:
pgd_acc_pgd_def = accuracy_score(y_true_pgd_pgd_def, y_pred_pgd_pgd_def)
pgd_prec_pgd_def = precision_score(y_true_pgd_pgd_def, y_pred_pgd_pgd_def)
pgd_rec_pgd_def = recall_score(y_true_pgd_pgd_def, y_pred_pgd_pgd_def)
pgd_f1_pgd_def = f1_score(y_true_pgd_pgd_def, y_pred_pgd_pgd_def)
cm_pgd_pgd_def = confusion_matrix(y_true_pgd_pgd_def, y_pred_pgd_pgd_def)

print("PGD-defended model on PGD Accuracy:", pgd_acc_pgd_def)
print("PGD-defended model on PGD Precision:", pgd_prec_pgd_def)
print("PGD-defended model on PGD Recall:", pgd_rec_pgd_def)
print("PGD-defended model on PGD F1-score:", pgd_f1_pgd_def)
print(cm_pgd_pgd_def)

PGD-defended model on PGD Accuracy: 0.7821593328601846
PGD-defended model on PGD Precision: 0.9222814498933902
PGD-defended model on PGD Recall: 0.6741214057507987
PGD-defended model on PGD F1-score: 0.7789132489983344
[[8982  729]
 [4182 8651]]


In [96]:
attack_mask = (y_true_pgd_pgd_def == 1)
pgd_esr_pgd_def = ((y_pred_pgd_pgd_def[attack_mask] == 0).sum() / attack_mask.sum()) * 100
print("PGD-defended model PGD ESR:", pgd_esr_pgd_def)

PGD-defended model PGD ESR: 32.587859424920126


In [97]:
final_results = pd.DataFrame([
    {
        "Condition": "Baseline on clean",
        "Accuracy": test_accuracy,
        "Precision": 0.9323952602623783,
        "Recall": 0.6867451102626042,
        "F1-score": 0.7909356069104779,
        "ESR": None
    },
    {
        "Condition": "Baseline on FGSM",
        "Accuracy": 0.7679657567423705,
        "Precision": 0.9188980716253443,
        "Recall": 0.6498090859502844,
        "F1-score": 0.7612744203030857,
        "ESR": 35.0109140497156
    },
    {
        "Condition": "Baseline on PGD",
        "Accuracy": 0.7072391767201916,
        "Precision": 0.9367904695164682,
        "Recall": 0.520844697264864,
        "F1-score": 0.6694711538461539,
        "ESR": 47.915530273513596
    },
    {
        "Condition": "FGSM-defended on FGSM",
        "Accuracy": fgsm_acc_def,
        "Precision": fgsm_prec_def,
        "Recall": fgsm_rec_def,
        "F1-score": fgsm_f1_def,
        "ESR": fgsm_esr_def
    },
    {
        "Condition": "FGSM-defended on PGD",
        "Accuracy": pgd_acc_def,
        "Precision": pgd_prec_def,
        "Recall": pgd_rec_def,
        "F1-score": pgd_f1_def,
        "ESR": pgd_esr_def
    },
    {
        "Condition": "PGD-defended on clean",
        "Accuracy": clean_acc_pgd_def,
        "Precision": clean_prec_pgd_def,
        "Recall": clean_rec_pgd_def,
        "F1-score": clean_f1_pgd_def,
        "ESR": None
    },
    {
        "Condition": "PGD-defended on FGSM",
        "Accuracy": fgsm_acc_pgd_def,
        "Precision": fgsm_prec_pgd_def,
        "Recall": fgsm_rec_pgd_def,
        "F1-score": fgsm_f1_pgd_def,
        "ESR": fgsm_esr_pgd_def
    },
    {
        "Condition": "PGD-defended on PGD",
        "Accuracy": pgd_acc_pgd_def,
        "Precision": pgd_prec_pgd_def,
        "Recall": pgd_rec_pgd_def,
        "F1-score": pgd_f1_pgd_def,
        "ESR": pgd_esr_pgd_def
    }
])

final_results

,Condition,Accuracy,Precision,Recall,F1-score,ESR
0,Baseline on clean,0.728220,0.932395,0.686745,0.790936,NaN
1,Baseline on FGSM,0.767966,0.918898,0.649809,0.761274,35.010914
2,Baseline on PGD,0.707239,0.936790,0.520845,0.669471,47.915530
3,FGSM-defended on FGSM,0.834368,0.934320,0.762643,0.839797,23.735681
4,FGSM-defended on PGD,0.827271,0.934227,0.749318,0.831618,25.068184
5,PGD-defended on clean,0.768320,0.918776,0.650510,0.761714,NaN
6,PGD-defended on FGSM,0.765969,0.917007,0.647471,0.759021,35.252864
7,PGD-defended on PGD,0.782159,0.922281,0.674121,0.778913,32.587859
